In [ ]:
import pandas as pd
import numpy as np
import json

# FILE CONFIGURATION
FORM_NAME = 'FV - X'
JSON_FILE = 'sampled_events_seed_X.json'
HUMAN_CSV = 'FV - X.csv'
JUDGE_CSV = '../data/judged_log.csv'
ORIGINAL_DATASET = '../data/forensic_atomic.csv' # F-Atomic dataset

# Columns to find the exact row in the original dataset
BRIDGE_COLS = ['event', 'xIntent', 'xEffect']

# JSON LOADING AND PARSING
with open(JSON_FILE, 'r', encoding='utf-8') as f:
    sample_data = json.load(f)

events_data = []
for e in sample_data['events']:
    # Get the info needed as a bridge
    event_info = {col: e.get(col, '') for col in BRIDGE_COLS}
    events_data.append(event_info)

print(f"Loaded {len(events_data)} events from JSON.")

# HUMAN RESPONSES PARSING
human_df = pd.read_csv(HUMAN_CSV)
records = []

for respondent_idx, row in human_df.iterrows():
    for i, event_info in enumerate(events_data):
        # In the CSV Form, column 0 = Timestamp. Event i occupies the next 3 columns.
        base_col = 1 + (i * 3)

        if base_col + 1 < len(human_df.columns):
            score_val = row.iloc[base_col]
            decision_val = str(row.iloc[base_col + 1]).upper()

            record = event_info.copy()
            record.update({
                'respondent': respondent_idx,
                'human_score': float(score_val) if pd.notna(score_val) else np.nan,
                'human_approved_int': 1 if 'APPROVE' in decision_val else 0
            })
            records.append(record)

human_parsed = pd.DataFrame(records)

# Aggregate by calculating human means for each unique event
human_agg = human_parsed.groupby(BRIDGE_COLS).agg({
    'human_score': 'mean',
    'human_approved_int': lambda x: (x.sum() / len(x)) * 100
}).reset_index()

human_agg.rename(columns={'human_approved_int': 'human_approval_pct'}, inplace=True)
print("Human parsing and aggregation completed.")

# RETRIEVE 'original_row_id' FROM ORIGINAL DATASET
original_df = pd.read_csv(ORIGINAL_DATASET)
original_df['original_row_id'] = original_df.index

# Ensure there are no blank spaces ruining the match
for col in BRIDGE_COLS:
    human_agg[col] = human_agg[col].astype(str).str.strip()
    original_df[col] = original_df[col].astype(str).str.strip()

# Retrieve the original ID by merging
human_with_id = pd.merge(human_agg, original_df[BRIDGE_COLS + ['original_row_id']], on=BRIDGE_COLS, how='inner')
print(f"Found {len(human_with_id)} exact original_row_ids from the original dataset.")

# MERGE WITH LLM JUDGES LOG
judge_df = pd.read_csv(JUDGE_CSV)

# Use the unique ID to merge humans and LLM
comparison = pd.merge(human_with_id, judge_df, on='original_row_id', how='inner', suffixes=('_human', '_llm'))

# Calculate comparison metrics
comparison['form_id'] = FORM_NAME
comparison['human_is_approved'] = (comparison['human_approval_pct'] >= 50) & (comparison['human_score'] >= 60)
comparison['llm_is_approved'] = comparison['status'] == 'APPROVED'
comparison['score_diff_abs'] = abs(comparison['human_score'] - comparison['score_avg_all'])
comparison['decision_agreement'] = comparison['human_is_approved'] == comparison['llm_is_approved']

print(f"Final merge with judges completed: {len(comparison)} merged events.")

# EXPORT
# Select only important columns for future statistics
export_cols = [
    'form_id', 'original_row_id', 'event_human',
    'human_score', 'human_approval_pct', 'human_is_approved',
    'score_avg_all', 'status', 'llm_is_approved',
    'score_diff_abs', 'decision_agreement'
]

# Rename 'event_human' to 'event' for cleanliness
final_export = comparison[export_cols].rename(columns={'event_human': 'event'})

export_filename = f'comparison_results_{FORM_NAME}.csv'
final_export.to_csv(export_filename, index=False)

print(f"Data saved in: {export_filename}")

In [ ]:
import pandas as pd
import glob
from scipy.stats import pearsonr, spearmanr
import json

# Find all generated files matching the pattern
file_list = glob.glob('comparison_results_FV_*.csv')

print(f"Found {len(file_list)} files to merge:")
for f in file_list:
    print(f"  - {f}")

# Load and concatenate all dataframes into one
df_list = []
for file in file_list:
    df = pd.read_csv(file)
    df_list.append(df)

global_df = pd.concat(df_list, ignore_index=True)
print(f"\nMerge completed, total analyzed events: {len(global_df)}")

# Save the global dataset for future analysis
global_csv_name = '../data/form_results/global_comparison_results.csv'
global_df.to_csv(global_csv_name, index=False)
print(f"Global dataset saved in: '{global_csv_name}'")

# Remove any rows with null scores before correlations
valid_scores = global_df.dropna(subset=['human_score', 'score_avg_all'])

# Calculate global correlation between humans and LLM
pearson_corr, p_value_p = pearsonr(valid_scores['human_score'], valid_scores['score_avg_all'])
spearman_corr, p_value_s = spearmanr(valid_scores['human_score'], valid_scores['score_avg_all'])

# Print GLOBAL STATISTICS
print("\nGLOBAL STATISTICS (ALL FORMS)")
print(f"Total Evaluated Events:           {len(global_df)}")
print(f"Human Approval Rate:              {(global_df['human_is_approved'].mean() * 100):.1f}%")
print(f"LLM Approval Rate:                {(global_df['llm_is_approved'].mean() * 100):.1f}%")
print(f"Decision Agreement (Appr/Rej):    {(global_df['decision_agreement'].mean() * 100):.1f}%")
print(f"Global Mean Score (Humans):       {global_df['human_score'].mean():.1f} / 100")
print(f"Global Mean Score (LLM):          {global_df['score_avg_all'].mean():.1f} / 100")
print(f"Mean Absolute Score Difference:   {global_df['score_diff_abs'].mean():.1f} points")
print(f"Pearson Correlation (r):          {pearson_corr:.3f} (p-value: {p_value_p:.4f})")
print(f"Spearman Correlation (rho):       {spearman_corr:.3f} (p-value: {p_value_s:.4f})")

# Optional: Export statistics to a JSON file
stats_dict = {
    "total_events": len(global_df),
    "human_approval_pct": round(global_df['human_is_approved'].mean() * 100, 1),
    "llm_approval_pct": round(global_df['llm_is_approved'].mean() * 100, 1),
    "decision_agreement_pct": round(global_df['decision_agreement'].mean() * 100, 1),
    "mean_human_score": round(global_df['human_score'].mean(), 1),
    "mean_llm_score": round(global_df['score_avg_all'].mean(), 1),
    "mean_absolute_diff": round(global_df['score_diff_abs'].mean(), 1),
    "pearson_r": round(pearson_corr, 3),
    "spearman_rho": round(spearman_corr, 3)
}

with open('../data/form_results/global_statistics_summary.json', 'w') as f:
    json.dump(stats_dict, f, indent=4)
print("\nStatistics also exported to 'global_statistics_summary.json'")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

# Load global data
df = pd.read_csv('../data/form_results/global_comparison_results.csv')
# Enforce approval logic (>=50% voters AND avg score >=60)
df['human_is_approved'] = (df['human_approval_pct'] >= 50) & (df['human_score'] >= 60)

# Calculate descriptive statistics
total = len(df)
agreement_rate = df['decision_agreement'].mean() * 100
human_app_rate = df['human_is_approved'].mean() * 100
llm_app_rate = df['llm_is_approved'].mean() * 100
avg_human_score = df['human_score'].mean()
avg_llm_score = df['score_avg_all'].mean()
mae = df['score_diff_abs'].mean()

# Calculate correlations (excluding NaNs)
valid = df.dropna(subset=['human_score', 'score_avg_all'])
p_r, p_val_p = pearsonr(valid['human_score'], valid['score_avg_all'])
s_r, p_val_s = spearmanr(valid['human_score'], valid['score_avg_all'])

# Visual Dashboard Setup
sns.set_style("whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
plt.suptitle(f"Global Analysis: Human vs LLM Validation ({total} Events)", fontsize=18, fontweight='bold', y=1.02)

# Pie Chart (Decision Agreement)
agreement_counts = df['decision_agreement'].value_counts().sort_index(ascending=False)
axes[0, 0].pie(agreement_counts, labels=['Agreement', 'Disagreement'], autopct='%1.1f%%',
               colors=['#4CAF50', '#F44336'], startangle=140, explode=(0.1, 0), shadow=True)
axes[0, 0].set_title('Decision Agreement (Approve/Reject)', fontweight='bold', fontsize=14)

# Bar Chart (Approval Rates)
rates = [human_app_rate, llm_app_rate]
labels = ['Humans', 'LLM Judge']
sns.barplot(x=labels, y=rates, palette=['#2196F3', '#FF9800'], ax=axes[0, 1], edgecolor='black')
axes[0, 1].set_ylim(0, 110)
axes[0, 1].set_ylabel('Approval Percentage (%)', fontsize=12)
axes[0, 1].set_title('Approval Rates Comparison', fontweight='bold', fontsize=14)
for i, v in enumerate(rates):
    axes[0, 1].text(i, v + 2, f"{v:.1f}%", ha='center', fontweight='bold', fontsize=12)

# Distribution (Score Density)
sns.kdeplot(df['human_score'], ax=axes[1, 0], fill=True, label=f'Humans (Mean: {avg_human_score:.1f})', color='#2196F3', alpha=0.5)
sns.kdeplot(df['score_avg_all'], ax=axes[1, 0], fill=True, label=f'LLM (Mean: {avg_llm_score:.1f})', color='#FF9800', alpha=0.5)
axes[1, 0].set_xlabel('Consistency Score (0-100)', fontsize=12)
axes[1, 0].set_title('Qualitative Score Distribution', fontweight='bold', fontsize=14)
axes[1, 0].legend(loc='upper left')

# Scatter Plot + Regression (Correlation)
sns.regplot(x='human_score', y='score_avg_all', data=df, ax=axes[1, 1],
            scatter_kws={'alpha':0.4, 'color':'purple', 's':50},
            line_kws={'color':'red', 'label': f'Pearson r: {p_r:.2f}'})
axes[1, 1].set_title('Correlation and Trend Line', fontweight='bold', fontsize=14)
axes[1, 1].set_xlabel('Human Score', fontsize=12)
axes[1, 1].set_ylabel('LLM Score', fontsize=12)
axes[1, 1].legend()

# Finalization
plt.tight_layout()
plt.savefig('thesis_validation_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()

# Print text summary for quick copy-paste
print("STATISTICAL SUMMARY FOR THESIS")
print(f"Analyzed Sample:          {total} events")
print(f"Decision Agreement:       {agreement_rate:.2f}%")
print(f"Human Approval Rate:      {human_app_rate:.1f}%")
print(f"LLM Approval Rate:        {llm_app_rate:.1f}%")
print(f"Mean Human Score:         {avg_human_score:.2f} / 100")
print(f"Mean LLM Score:           {avg_llm_score:.2f} / 100")
print(f"MAE (Mean Absolute Error):{mae:.2f} points")
print(f"Pearson r:                {p_r:.3f} (p-value: {p_val_p:.4e})")
print(f"Spearman rho:             {s_r:.3f} (p-value: {p_val_s:.4e})")

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

df = pd.read_csv('../data/form_results/global_comparison_results.csv')
valid_df = df.dropna(subset=['human_is_approved', 'llm_is_approved'])

# Human = Truth (y_true), LLM = Prediction (y_pred)
# True = 1, False = 0
y_true = valid_df['human_is_approved'].astype(int)
y_pred = valid_df['llm_is_approved'].astype(int)

# Calculate Confusion Matrix
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print(f"True Positive (TP) [Both OK]: {tp}")
print(f"False Positive (FP) [Human NO, LLM OK]: {fp}")
print(f"False Negative (FN) [Human OK, LLM NO]: {fn}")
print(f"True Negative (TN) [Both NO]: {tn}")